In [1]:
import pandas as pd
import math


data = {
    'Outlook': [
        'Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain',
        'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain',
        'Sunny', 'Overcast', 'Overcast', 'Rain'
    ],

    'Temperature': [
        'Hot', 'Hot', 'Hot', 'Mild', 'Cool',
        'Cool', 'Cool', 'Mild', 'Cool', 'Mild',
        'Mild', 'Mild', 'Hot', 'Mild'
    ],

    'Humidity': [
        'High', 'High', 'High', 'High', 'Normal',
        'Normal', 'Normal', 'High', 'Normal', 'Normal',
        'Normal', 'High', 'Normal', 'High'
    ],

    'Wind': [
        'Weak', 'Strong', 'Weak', 'Weak', 'Weak',
        'Strong', 'Strong', 'Weak', 'Weak', 'Weak',
        'Strong', 'Strong', 'Weak', 'Strong'
    ],

    'Play': [
        'No', 'No', 'Yes', 'Yes', 'Yes',
        'No', 'Yes', 'No', 'Yes', 'Yes',
        'Yes', 'Yes', 'Yes', 'No'
    ]
}

df = pd.DataFrame(data)

print("DATASET")
print(df)



def entropy(data, target):

    values = data[target].value_counts()

    total = len(data)

    ent = 0

    for count in values:

        probability = count / total

        ent -= probability * math.log2(probability)

    return ent


def information_gain(data, attribute, target):

    total_entropy = entropy(data, target)

    weighted_entropy = 0

    for value in data[attribute].unique():

        subset = data[data[attribute] == value]

        weight = len(subset) / len(data)

        weighted_entropy += weight * entropy(
            subset,
            target
        )

    gain = total_entropy - weighted_entropy

    return gain

def best_attribute(data, attributes, target):

    gains = {}

    for attribute in attributes:

        gains[attribute] = information_gain(
            data,
            attribute,
            target
        )

    print("\nInformation Gain:")

    for attribute, gain in gains.items():

        print(
            attribute,
            "=",
            round(gain, 4)
        )

    best = max(
        gains,
        key=gains.get
    )

    print("Best Attribute:", best)

    return best


def id3(data, attributes, target):

    # If all target values are same
    if len(data[target].unique()) == 1:

        return data[target].iloc[0]

    # If no attributes remain
    if len(attributes) == 0:

        return data[target].mode()[0]

    # Find best attribute
    best = best_attribute(
        data,
        attributes,
        target
    )

    tree = {
        best: {}
    }

    # Create branches
    for value in data[best].unique():

        subset = data[
            data[best] == value
        ]

        remaining_attributes = [
            attribute
            for attribute in attributes
            if attribute != best
        ]

        subtree = id3(
            subset,
            remaining_attributes,
            target
        )

        tree[best][value] = subtree

    return tree


attributes = [
    'Outlook',
    'Temperature',
    'Humidity',
    'Wind'
]

print("\n\nBUILDING DECISION TREE")

decision_tree = id3(
    df,
    attributes,
    'Play'
)

print("\nFINAL DECISION TREE:")
print(decision_tree)


def predict(tree, sample):

    # If tree has reached a leaf
    if not isinstance(tree, dict):

        return tree

    # Get decision attribute
    attribute = next(iter(tree))

    # Get sample value
    value = sample[attribute]

    # Check whether branch exists
    if value in tree[attribute]:

        return predict(
            tree[attribute][value],
            sample
        )

    else:

        return "Unknown"



sample1 = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

sample2 = {
    'Outlook': 'Overcast',
    'Temperature': 'Hot',
    'Humidity': 'High',
    'Wind': 'Strong'
}

sample3 = {
    'Outlook': 'Rain',
    'Temperature': 'Mild',
    'Humidity': 'Normal',
    'Wind': 'Weak'
}


print("\nPREDICTIONS")

print(
    "Sample 1:",
    predict(decision_tree, sample1)
)

print(
    "Sample 2:",
    predict(decision_tree, sample2)
)

print(
    "Sample 3:",
    predict(decision_tree, sample3)
)

DATASET
     Outlook Temperature Humidity    Wind Play
0      Sunny         Hot     High    Weak   No
1      Sunny         Hot     High  Strong   No
2   Overcast         Hot     High    Weak  Yes
3       Rain        Mild     High    Weak  Yes
4       Rain        Cool   Normal    Weak  Yes
5       Rain        Cool   Normal  Strong   No
6   Overcast        Cool   Normal  Strong  Yes
7      Sunny        Mild     High    Weak   No
8      Sunny        Cool   Normal    Weak  Yes
9       Rain        Mild   Normal    Weak  Yes
10     Sunny        Mild   Normal  Strong  Yes
11  Overcast        Mild     High  Strong  Yes
12  Overcast         Hot   Normal    Weak  Yes
13      Rain        Mild     High  Strong   No


BUILDING DECISION TREE

Information Gain:
Outlook = 0.2467
Temperature = 0.0292
Humidity = 0.1518
Wind = 0.0481
Best Attribute: Outlook

Information Gain:
Temperature = 0.571
Humidity = 0.971
Wind = 0.02
Best Attribute: Humidity

Information Gain:
Temperature = 0.02
Humidity = 0.02
Wi